# Preprocessing and Training Data Development

The purpose of this notebook is to prepare the Aspire Public Schools California School Dashboard dataset for machine learning modeling.

The goal of this project is to predict next year’s Dashboard color for each school and indicator using historical performance data.

This notebook includes:
- creating the next-year prediction target
- selecting modeling features
- creating dummy variables for categorical features
- scaling numeric features
- splitting the data into training and testing sets
- saving the processed datasets for modeling

In [6]:
# Import necessary libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [7]:
# Load cleaned Aspire Dashboard dataset

df = pd.read_csv("aspire_all.csv")

# Preview the dataset
df.head()

,cds,schoolname,indicator,year,studentgroup,currstatus,priorstatus,change,statuslevel,changelevel,color
0,1612590109819,Aspire Berkley Maynard Academy,chronic_absenteeism,2023,All Students,35.0,27.9,7.1,1,1,1
1,1612590118224,Aspire Golden State College Preparatory Academy,chronic_absenteeism,2023,All Students,40.1,40.8,-0.6,1,4,2
2,1612590128413,Aspire College Academy,chronic_absenteeism,2023,All Students,48.2,63.8,-15.6,1,5,3
3,1612590130666,Aspire Lionel Wilson College Preparatory Academy,chronic_absenteeism,2023,All Students,23.7,18.6,5.1,1,1,1
4,1612590130732,Aspire Triumph Technology Academy,chronic_absenteeism,2023,All Students,59.8,66.9,-7.1,1,5,3


## Create the Prediction Target: Next Year's Dashboard Color

To create a true predictive modeling problem, I created a new target variable called `next_year_color`.

This was done by sorting the data by school, indicator, and year, then shifting the `color` column forward one year within each school-indicator group.

This allows the model to use one year's performance metrics to predict the following year's Dashboard color.

In [10]:
# Sort dataset by school, indicator, and year

df = df.sort_values(by=['schoolname', 'indicator', 'year'])

# Create next year's Dashboard color target

df['next_year_color'] = (
    df.groupby(['schoolname', 'indicator'])['color']
    .shift(-1)
)

# Preview the shifted target variable

df[['schoolname', 'indicator', 'year', 'color', 'next_year_color']].head(12)

,schoolname,indicator,year,color,next_year_color
28,Aspire APEX Academy,chronic_absenteeism,2023,3,3.0
63,Aspire APEX Academy,chronic_absenteeism,2024,3,1.0
98,Aspire APEX Academy,chronic_absenteeism,2025,1,NaN
134,Aspire APEX Academy,ela_sbac,2023,2,1.0
170,Aspire APEX Academy,ela_sbac,2024,1,1.0
205,Aspire APEX Academy,ela_sbac,2025,1,NaN
241,Aspire APEX Academy,math_sbac,2023,3,1.0
277,Aspire APEX Academy,math_sbac,2024,1,1.0
312,Aspire APEX Academy,math_sbac,2025,1,NaN
348,Aspire APEX Academy,suspension_rate,2023,5,1.0


## Remove Rows Without a Future Target

Rows from the final year of data do not have a corresponding next-year Dashboard color available. Since supervised machine learning models require a known target variable, these rows are removed from the dataset.

The `next_year_color` variable is also converted to an integer data type for classification modeling.

In [11]:
# Remove rows where next_year_color is missing

df = df.dropna(subset=['next_year_color']).copy()

# Convert next_year_color to integer

df['next_year_color'] = df['next_year_color'].astype(int)

# Check dataframe structure

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 284 entries, 28 to 379
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   cds              284 non-null    int64  
 1   schoolname       284 non-null    object 
 2   indicator        284 non-null    object 
 3   year             284 non-null    int64  
 4   studentgroup     284 non-null    object 
 5   currstatus       284 non-null    float64
 6   priorstatus      284 non-null    float64
 7   change           284 non-null    float64
 8   statuslevel      284 non-null    int64  
 9   changelevel      284 non-null    int64  
 10  color            284 non-null    int64  
 11  next_year_color  284 non-null    int64  
dtypes: float64(3), int64(6), object(3)
memory usage: 28.8+ KB


In [12]:
# Examine distribution of the target variable

df['next_year_color'].value_counts().sort_index()

next_year_color
1    57
2    72
3    95
4    29
5    31
Name: count, dtype: int64

## Target Variable Distribution

The target variable, `next_year_color`, contains five Dashboard color classes. The class distribution is relatively balanced overall, although classes 4 and 5 contain fewer observations than classes 2 and 3.

This distribution is acceptable for multiclass classification modeling, though class imbalance should still be considered during model evaluation.

## Feature Selection

The target variable for modeling is `next_year_color`.

Several variables were excluded from the feature set to reduce data leakage and redundancy:
- `next_year_color` is the prediction target
- `color` represents the current-year Dashboard color and would leak information into the model
- `schoolname`, `studentgroup`, and `cds` were excluded because they function primarily as identifiers rather than predictive features

The selected features focus on historical performance metrics and indicator type.

In [13]:
# Select modeling features

features = [
    'priorstatus',
    'change',
    'year',
    'indicator'
]

# Create modeling dataframe

model_df = df[features + ['next_year_color']].copy()

# Preview modeling dataframe

model_df.head()

,priorstatus,change,year,indicator,next_year_color
28,53.7,-8.7,2023,chronic_absenteeism,3
63,45.0,-13.0,2024,chronic_absenteeism,1
134,-90.4,7.0,2023,ela_sbac,1
170,-83.4,-7.8,2024,ela_sbac,1
241,-119.7,27.2,2023,math_sbac,1


## Feature Engineering

To help capture whether a school's performance improved from the prior year, I created a binary feature called `improved`.

- A value of 1 indicates the school's performance improved (`change > 0`)
- A value of 0 indicates the school's performance stayed the same or declined

This feature may help the model better capture directional changes in school performance.

In [14]:
# Create improved feature

model_df['improved'] = (model_df['change'] > 0).astype(int)

# Preview updated dataframe

model_df.head()

,priorstatus,change,year,indicator,next_year_color,improved
28,53.7,-8.7,2023,chronic_absenteeism,3,0
63,45.0,-13.0,2024,chronic_absenteeism,1,0
134,-90.4,7.0,2023,ela_sbac,1,1
170,-83.4,-7.8,2024,ela_sbac,1,0
241,-119.7,27.2,2023,math_sbac,1,1


## Create Dummy Variables for Categorical Features

Machine learning models require numeric input variables. Therefore, the categorical variable `indicator` was converted into dummy variables using one-hot encoding.

I used `drop_first=True` to avoid redundant dummy variables and reduce multicollinearity.

In [15]:
# Create dummy variables for indicator

model_df = pd.get_dummies(
    model_df,
    columns=['indicator'],
    drop_first=True
)

# Preview encoded dataframe

model_df.head()

,priorstatus,change,year,next_year_color,improved,indicator_ela_sbac,indicator_math_sbac,indicator_suspension_rate
28,53.7,-8.7,2023,3,0,False,False,False
63,45.0,-13.0,2024,1,0,False,False,False
134,-90.4,7.0,2023,1,1,True,False,False
170,-83.4,-7.8,2024,1,0,True,False,False
241,-119.7,27.2,2023,1,1,False,True,False


In [16]:
# Confirm all columns are numeric after encoding

model_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 284 entries, 28 to 379
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   priorstatus                284 non-null    float64
 1   change                     284 non-null    float64
 2   year                       284 non-null    int64  
 3   next_year_color            284 non-null    int64  
 4   improved                   284 non-null    int64  
 5   indicator_ela_sbac         284 non-null    bool   
 6   indicator_math_sbac        284 non-null    bool   
 7   indicator_suspension_rate  284 non-null    bool   
dtypes: bool(3), float64(2), int64(3)
memory usage: 14.1 KB


## Define Features and Target Variable

The feature matrix `X` contains all predictor variables used for modeling.

The target vector `y` contains the next year's Dashboard color classification.

In [17]:
# Define feature matrix and target vector

X = model_df.drop('next_year_color', axis=1)

y = model_df['next_year_color']

# Check shapes

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (284, 7)
y shape: (284,)


## Split Data Into Training and Testing Sets

The dataset was split into training and testing subsets so model performance can later be evaluated on unseen data.

I used stratification to preserve the distribution of Dashboard color classes in both the training and testing sets.

In [18]:
# Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Print dataset shapes

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (213, 7)
X_test shape: (71, 7)
y_train shape: (213,)
y_test shape: (71,)


In [19]:
# Examine target distribution in training set

y_train.value_counts(normalize=True).sort_index()

next_year_color
1    0.201878
2    0.253521
3    0.333333
4    0.103286
5    0.107981
Name: proportion, dtype: float64

In [20]:
# Examine target distribution in testing set

y_test.value_counts(normalize=True).sort_index()

next_year_color
1    0.197183
2    0.253521
3    0.338028
4    0.098592
5    0.112676
Name: proportion, dtype: float64

## Review Class Distribution After Splitting

The class distributions in the training and testing datasets are very similar, confirming that stratified sampling successfully preserved the proportion of each Dashboard color class.

Maintaining similar class distributions helps ensure the model is evaluated fairly and reduces the risk of biased performance metrics.

## Standardize Numeric Features

The numeric variables in the dataset exist on different scales. For example, `priorstatus` contains large negative and positive values, while binary features such as `improved` range only from 0 to 1.

To ensure numeric variables contribute more evenly during model training, I standardized the numeric features using `StandardScaler`.

The scaler was fit only on the training data and then applied to the testing data to prevent data leakage.

In [21]:
# Identify numeric columns to scale

numeric_cols = ['priorstatus', 'change', 'year']

# Create scaler object

scaler = StandardScaler()

# Create copies of training and testing data

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Fit scaler on training data and transform training data

X_train_scaled[numeric_cols] = scaler.fit_transform(
    X_train_scaled[numeric_cols]
)

# Transform testing data using training scaler

X_test_scaled[numeric_cols] = scaler.transform(
    X_test_scaled[numeric_cols]
)

In [22]:
# Preview scaled training data

X_train_scaled.head()

,priorstatus,change,year,improved,indicator_ela_sbac,indicator_math_sbac,indicator_suspension_rate
47,1.258168,-1.021617,0.995316,0,False,False,False
390,0.546665,0.121289,0.995316,0,False,False,True
379,0.483548,0.121289,0.995316,0,False,False,True
67,1.307897,-0.939393,0.995316,0,False,False,False
118,-0.935632,0.762632,-1.004706,1,True,False,False


In [23]:
# Confirm no missing values exist after scaling

print("Missing values in X_train_scaled:")
print(X_train_scaled.isnull().sum())

print("\nMissing values in X_test_scaled:")
print(X_test_scaled.isnull().sum())

Missing values in X_train_scaled:
priorstatus                  0
change                       0
year                         0
improved                     0
indicator_ela_sbac           0
indicator_math_sbac          0
indicator_suspension_rate    0
dtype: int64

Missing values in X_test_scaled:
priorstatus                  0
change                       0
year                         0
improved                     0
indicator_ela_sbac           0
indicator_math_sbac          0
indicator_suspension_rate    0
dtype: int64


## Save Preprocessed Datasets

The final preprocessed training and testing datasets were saved as CSV files so they can be reused during the modeling phase of the project.

Saving intermediate datasets improves reproducibility and allows the modeling notebook to remain focused on machine learning development and evaluation.

In [24]:
# Save processed training and testing datasets

X_train_scaled.to_csv("X_train_scaled.csv", index=False)
X_test_scaled.to_csv("X_test_scaled.csv", index=False)

y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

# Conclusion

In this notebook, I prepared the Aspire Dashboard dataset for machine learning modeling by:

- creating a future Dashboard color prediction target,
- engineering additional predictive features,
- converting categorical variables into dummy variables,
- splitting the dataset into training and testing subsets,
- standardizing numeric variables,
- and saving the final processed datasets.

The resulting datasets are now ready for the modeling phase of the project, where classification algorithms will be used to predict next year’s Dashboard color classifications for Aspire schools.